# World Cup Prediction Model — Best Current Workflow

This notebook keeps only the relevant current pipeline for building the strongest version of the model:

1. Load and normalize data.
2. Build final dynamic football features: smarter Elo, recent form, weighted attack/defense, tournament importance, confederation strength, and host advantage.
3. Train final goal models with chronological validation, recency weighting, and LightGBM fallback.
4. Calibrate win/draw/loss probabilities.
5. Simulate the 2026 World Cup with calibrated score sampling and official-style knockout bracket logic.

Monte Carlo is intentionally kept at **100 simulations** while iterating. Increase it later once the notebook is stable.

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np

from pathlib import Path
from itertools import combinations
from collections import Counter, defaultdict
from scipy.stats import poisson
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import mean_absolute_error, log_loss, brier_score_loss
from sklearn.isotonic import IsotonicRegression
from tqdm import tqdm

DATA_DIR = Path("data")

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

# Main modeling settings
TRAIN_START_DATE = "2000-01-01"
SPLIT_DATE = "2022-01-01"
RECENCY_HALFLIFE_DAYS = 365 * 3
N_SIMULATIONS = 100

# Final feature set used by the model.
FEATURES = [
    "elo_diff",
    "form_diff",
    "attack_diff",
    "defense_diff",
    "neutral",
    "importance",
    "conf_strength_diff",
    "host_diff",
]

## 1. Load Data

In [ ]:
fifa = pd.read_csv(DATA_DIR / "fifa_mens_rankings.csv")
elo_current = pd.read_csv(DATA_DIR / "world_football_elo_ratings.csv")
results_raw = pd.read_csv(DATA_DIR / "results.csv")

# Optional files. The current model does not need these yet, but loading them keeps the project ready for player/squad features later.
goalscorers = pd.read_csv(DATA_DIR / "goalscorers.csv") if (DATA_DIR / "goalscorers.csv").exists() else None
shootouts = pd.read_csv(DATA_DIR / "shootouts.csv") if (DATA_DIR / "shootouts.csv").exists() else None
former_names = pd.read_csv(DATA_DIR / "former_names.csv") if (DATA_DIR / "former_names.csv").exists() else None

print("fifa:", fifa.shape)
print("elo_current:", elo_current.shape)
print("results_raw:", results_raw.shape)
display(results_raw.head())

## 2. Team Name Normalization

In [ ]:
def clean_team(name):
    if pd.isna(name):
        return name

    replacements = {
        "United States": "USA",
        "USMNT": "USA",
        "Korea Republic": "South Korea",
        "IR Iran": "Iran",
        "Türkiye": "Turkey",
        "Côte d'Ivoire": "Ivory Coast",
        "Curaçao": "Curacao",
    }

    value = str(name).strip()
    return replacements.get(value, value)

for df, cols in [
    (fifa, ["team"]),
    (elo_current, ["team"]),
    (results_raw, ["home_team", "away_team", "country"]),
]:
    for col in cols:
        if col in df.columns:
            df[col] = df[col].apply(clean_team)

## 3. Modeling Constants

In [ ]:
INITIAL_ELO = 1500
HOST_2026 = {"USA", "Mexico", "Canada"}

# Simple manual confederation strength prior. This can be replaced later with a data-driven confederation rating.
CONFEDERATION_STRENGTH = {
    # UEFA
    "Spain": 1.00, "France": 1.00, "England": 1.00, "Germany": 1.00, "Portugal": 1.00,
    "Netherlands": 1.00, "Belgium": 1.00, "Croatia": 1.00, "Italy": 1.00, "Denmark": 1.00,
    "Switzerland": 1.00, "Austria": 1.00, "Norway": 1.00, "Poland": 1.00, "Serbia": 1.00,
    "Scotland": 1.00, "Slovakia": 1.00, "Turkey": 1.00, "Ukraine": 1.00, "Sweden": 1.00,
    "Wales": 1.00, "Czech Republic": 1.00, "Hungary": 1.00, "Romania": 1.00,
    # CONMEBOL
    "Argentina": 0.97, "Brazil": 0.97, "Uruguay": 0.97, "Colombia": 0.97, "Ecuador": 0.97,
    "Paraguay": 0.97, "Chile": 0.97, "Peru": 0.97, "Bolivia": 0.97, "Venezuela": 0.97,
    # CAF
    "Morocco": 0.88, "Senegal": 0.88, "Egypt": 0.88, "Ghana": 0.88, "Ivory Coast": 0.88,
    "Nigeria": 0.88, "Algeria": 0.88, "Tunisia": 0.88, "South Africa": 0.88, "Cape Verde": 0.88,
    "Cameroon": 0.88, "Mali": 0.88,
    # AFC
    "Japan": 0.84, "South Korea": 0.84, "Iran": 0.84, "Australia": 0.84, "Saudi Arabia": 0.84,
    "Qatar": 0.84, "Uzbekistan": 0.84, "Jordan": 0.84, "Iraq": 0.84,
    # CONCACAF
    "USA": 0.82, "Mexico": 0.82, "Canada": 0.82, "Panama": 0.82, "Jamaica": 0.82,
    "Costa Rica": 0.82, "Haiti": 0.82, "Curacao": 0.82, "Honduras": 0.82,
    # OFC
    "New Zealand": 0.72,
}
DEFAULT_CONFEDERATION_STRENGTH = 0.85


def get_conf_strength(team):
    return CONFEDERATION_STRENGTH.get(team, DEFAULT_CONFEDERATION_STRENGTH)


def tournament_importance(tournament):
    if pd.isna(tournament):
        return 1

    t = str(tournament).lower()

    if "fifa world cup" in t and "qualification" not in t:
        return 5
    if "uefa euro" in t and "qualification" not in t:
        return 4
    if "copa américa" in t or "copa america" in t:
        return 4
    if "african cup" in t or "asian cup" in t or "gold cup" in t:
        return 3
    if "nations league" in t:
        return 3
    if "qualification" in t or "qualifier" in t:
        return 2
    if "friendly" in t:
        return 1

    return 2

## 4. Final Feature Engineering: Dynamic Elo + Weighted Form + Attack/Defense

In [ ]:
def expected_score(rating_a, rating_b):
    return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))


def elo_k_factor(tournament, goal_diff):
    importance = tournament_importance(tournament)
    base_k = {
        1: 15,
        2: 25,
        3: 35,
        4: 45,
        5: 60,
    }.get(importance, 25)

    goal_diff_multiplier = 1 + np.log1p(abs(goal_diff)) / 2
    return base_k * goal_diff_multiplier


def update_elo(home_rating, away_rating, home_score, away_score, tournament):
    expected_home = expected_score(home_rating, away_rating)
    expected_away = expected_score(away_rating, home_rating)

    if home_score > away_score:
        actual_home, actual_away = 1, 0
    elif home_score < away_score:
        actual_home, actual_away = 0, 1
    else:
        actual_home, actual_away = 0.5, 0.5

    k = elo_k_factor(tournament, home_score - away_score)
    return (
        home_rating + k * (actual_home - expected_home),
        away_rating + k * (actual_away - expected_away),
    )


def match_points(goals_for, goals_against):
    if goals_for > goals_against:
        return 3
    if goals_for == goals_against:
        return 1
    return 0


def weighted_recent_average(values, window=20, default=1.3, decay=10):
    values = list(values)[-window:]
    if len(values) == 0:
        return default

    # Most recent observations get the largest weights.
    weights = np.exp(-np.arange(len(values))[::-1] / decay)
    return float(np.average(values, weights=weights))


def historical_host_flags(row):
    country = row["country"] if "country" in row.index else None
    home = row["home_team"]
    away = row["away_team"]

    home_host = int(pd.notna(country) and clean_team(country) == home)
    away_host = int(pd.notna(country) and clean_team(country) == away)
    return home_host, away_host


results = results_raw.copy()
results["date"] = pd.to_datetime(results["date"])
results = results.sort_values("date").reset_index(drop=True)
results = results.dropna(subset=["home_score", "away_score"]).copy()
results["neutral"] = results["neutral"].astype(int)

team_elos = defaultdict(lambda: INITIAL_ELO)
team_points = defaultdict(list)
goals_for = defaultdict(list)
goals_against = defaultdict(list)

dynamic_rows = []

for _, row in results.iterrows():
    home = row["home_team"]
    away = row["away_team"]
    home_score = row["home_score"]
    away_score = row["away_score"]
    tournament = row.get("tournament", None)

    home_elo_before = team_elos[home]
    away_elo_before = team_elos[away]

    home_form_10 = weighted_recent_average(team_points[home], window=10, default=1.5, decay=5)
    away_form_10 = weighted_recent_average(team_points[away], window=10, default=1.5, decay=5)

    home_attack = weighted_recent_average(goals_for[home], window=20, default=1.3, decay=10)
    home_defense = weighted_recent_average(goals_against[home], window=20, default=1.3, decay=10)
    away_attack = weighted_recent_average(goals_for[away], window=20, default=1.3, decay=10)
    away_defense = weighted_recent_average(goals_against[away], window=20, default=1.3, decay=10)

    home_conf_strength = get_conf_strength(home)
    away_conf_strength = get_conf_strength(away)
    home_host, away_host = historical_host_flags(row)

    dynamic_rows.append({
        "home_elo_before": home_elo_before,
        "away_elo_before": away_elo_before,
        "elo_diff": home_elo_before - away_elo_before,
        "home_form_10": home_form_10,
        "away_form_10": away_form_10,
        "form_diff": home_form_10 - away_form_10,
        "home_attack": home_attack,
        "home_defense": home_defense,
        "away_attack": away_attack,
        "away_defense": away_defense,
        "attack_diff": home_attack - away_attack,
        "defense_diff": away_defense - home_defense,
        "home_conf_strength": home_conf_strength,
        "away_conf_strength": away_conf_strength,
        "conf_strength_diff": home_conf_strength - away_conf_strength,
        "home_host": home_host,
        "away_host": away_host,
        "host_diff": home_host - away_host,
    })

    new_home_elo, new_away_elo = update_elo(
        home_elo_before,
        away_elo_before,
        home_score,
        away_score,
        tournament,
    )
    team_elos[home] = new_home_elo
    team_elos[away] = new_away_elo

    team_points[home].append(match_points(home_score, away_score))
    team_points[away].append(match_points(away_score, home_score))

    goals_for[home].append(home_score)
    goals_against[home].append(away_score)
    goals_for[away].append(away_score)
    goals_against[away].append(home_score)

matches = pd.concat([results.reset_index(drop=True), pd.DataFrame(dynamic_rows)], axis=1)
matches["importance"] = matches["tournament"].apply(tournament_importance)

team_strength_dynamic = pd.DataFrame([
    {
        "team": team,
        "dynamic_elo": rating,
        "form_10": weighted_recent_average(team_points[team], window=10, default=1.5, decay=5),
        "attack_20": weighted_recent_average(goals_for[team], window=20, default=1.3, decay=10),
        "defense_20": weighted_recent_average(goals_against[team], window=20, default=1.3, decay=10),
        "conf_strength": get_conf_strength(team),
        "is_2026_host": int(team in HOST_2026),
    }
    for team, rating in team_elos.items()
])

matches.to_csv(DATA_DIR / "match_training_data_final.csv", index=False)
team_strength_dynamic.to_csv(DATA_DIR / "team_strength_dynamic_final.csv", index=False)

print("matches:", matches.shape)
print("team_strength_dynamic:", team_strength_dynamic.shape)
display(team_strength_dynamic.sort_values("dynamic_elo", ascending=False).head(20))

## 5. Train Final Goal Models with Chronological Holdout

In [ ]:
model_data = matches.dropna(subset=FEATURES + ["home_score", "away_score"]).copy()
model_data = model_data[model_data["date"] >= TRAIN_START_DATE].copy()
model_data = model_data.sort_values("date").reset_index(drop=True)

for col in FEATURES:
    model_data[col] = pd.to_numeric(model_data[col], errors="coerce")

model_data = model_data.dropna(subset=FEATURES + ["home_score", "away_score"]).copy()

# Recency + tournament-importance sample weighting.
days_old = (model_data["date"].max() - model_data["date"]).dt.days
model_data["recency_weight"] = np.exp(-np.log(2) * days_old / RECENCY_HALFLIFE_DAYS)
model_data["importance_weight"] = model_data["importance"] / model_data["importance"].mean()
model_data["sample_weight"] = model_data["recency_weight"] * model_data["importance_weight"]

train_df = model_data[model_data["date"] < SPLIT_DATE].copy()
test_df = model_data[model_data["date"] >= SPLIT_DATE].copy()

if len(train_df) == 0 or len(test_df) == 0:
    raise ValueError("Chronological split produced an empty train or test set. Adjust SPLIT_DATE.")

X_train = train_df[FEATURES].astype(float)
X_test = test_df[FEATURES].astype(float)

y_home_train = train_df["home_score"]
y_home_test = test_df["home_score"]
y_away_train = train_df["away_score"]
y_away_test = test_df["away_score"]

sample_weight_train = train_df["sample_weight"]

try:
    from lightgbm import LGBMRegressor

    home_model = LGBMRegressor(
        objective="poisson",
        n_estimators=1000,
        learning_rate=0.02,
        max_depth=6,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_SEED,
        verbose=-1,
    )
    away_model = LGBMRegressor(
        objective="poisson",
        n_estimators=1000,
        learning_rate=0.02,
        max_depth=6,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_SEED,
        verbose=-1,
    )
    model_type = "LightGBM"

except Exception as e:
    print("LightGBM unavailable. Using PoissonRegressor fallback.")
    print(type(e).__name__, e)
    home_model = PoissonRegressor(alpha=1.0, max_iter=1000)
    away_model = PoissonRegressor(alpha=1.0, max_iter=1000)
    model_type = "PoissonRegressor fallback"

home_model.fit(X_train, y_home_train, sample_weight=sample_weight_train)
away_model.fit(X_train, y_away_train, sample_weight=sample_weight_train)

home_pred = np.clip(home_model.predict(X_test), 0.05, None)
away_pred = np.clip(away_model.predict(X_test), 0.05, None)

print("Model:", model_type)
print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print("Home MAE:", mean_absolute_error(y_home_test, home_pred))
print("Away MAE:", mean_absolute_error(y_away_test, away_pred))

## 6. Probability Helpers + Calibration

In [ ]:
def match_probabilities(home_lambda, away_lambda, max_goals=10):
    home_lambda = max(float(home_lambda), 0.05)
    away_lambda = max(float(away_lambda), 0.05)

    home_win = 0.0
    draw = 0.0
    away_win = 0.0
    score_probs = []

    for h in range(max_goals + 1):
        for a in range(max_goals + 1):
            p = poisson.pmf(h, home_lambda) * poisson.pmf(a, away_lambda)
            result = int(np.sign(h - a))
            score_probs.append({
                "home_score": h,
                "away_score": a,
                "probability": p,
                "result": result,
            })

            if h > a:
                home_win += p
            elif h == a:
                draw += p
            else:
                away_win += p

    total = home_win + draw + away_win
    return {
        "home_win": home_win / total,
        "draw": draw / total,
        "away_win": away_win / total,
        "score_probs": pd.DataFrame(score_probs),
    }


def normalize_three_probs(home_p, draw_p, away_p):
    values = np.array([home_p, draw_p, away_p], dtype=float)
    values = np.clip(values, 1e-6, 1.0)
    values = values / values.sum()
    return {"home_win": float(values[0]), "draw": float(values[1]), "away_win": float(values[2])}


def result_from_scores(home_score, away_score):
    return int(np.sign(home_score - away_score))


def predict_lambdas_from_features(X_row):
    X_row = X_row[FEATURES].astype(float)
    home_lambda = max(float(home_model.predict(X_row)[0]), 0.05)
    away_lambda = max(float(away_model.predict(X_row)[0]), 0.05)
    return home_lambda, away_lambda


# Calibrate W/D/L probabilities using training-period predictions only.
calibration_rows = []

for _, row in train_df.iterrows():
    X_row = pd.DataFrame([row[FEATURES].to_dict()]).apply(pd.to_numeric, errors="coerce").astype(float)
    home_lambda, away_lambda = predict_lambdas_from_features(X_row)
    probs = match_probabilities(home_lambda, away_lambda)
    actual = result_from_scores(row["home_score"], row["away_score"])

    calibration_rows.append({
        "home_win_probability": probs["home_win"],
        "draw_probability": probs["draw"],
        "away_win_probability": probs["away_win"],
        "actual_home_win": int(actual == 1),
        "actual_draw": int(actual == 0),
        "actual_away_win": int(actual == -1),
    })

calibration_df = pd.DataFrame(calibration_rows)

home_win_calibrator = IsotonicRegression(out_of_bounds="clip")
draw_calibrator = IsotonicRegression(out_of_bounds="clip")
away_win_calibrator = IsotonicRegression(out_of_bounds="clip")

home_win_calibrator.fit(calibration_df["home_win_probability"], calibration_df["actual_home_win"])
draw_calibrator.fit(calibration_df["draw_probability"], calibration_df["actual_draw"])
away_win_calibrator.fit(calibration_df["away_win_probability"], calibration_df["actual_away_win"])


def calibrate_result_probabilities(raw_probs):
    calibrated_home = home_win_calibrator.predict([raw_probs["home_win"]])[0]
    calibrated_draw = draw_calibrator.predict([raw_probs["draw"]])[0]
    calibrated_away = away_win_calibrator.predict([raw_probs["away_win"]])[0]
    return normalize_three_probs(calibrated_home, calibrated_draw, calibrated_away)


def calibrated_score_distribution(home_lambda, away_lambda, max_goals=10):
    """Score distribution whose W/D/L mass is adjusted to calibrated probabilities.

    This lets the tournament simulator keep exact scores for group tables while still using calibrated
    outcome probabilities.
    """
    raw = match_probabilities(home_lambda, away_lambda, max_goals=max_goals)
    calibrated = calibrate_result_probabilities(raw)
    scores = raw["score_probs"].copy()

    raw_mass_by_result = {
        1: raw["home_win"],
        0: raw["draw"],
        -1: raw["away_win"],
    }
    calibrated_mass_by_result = {
        1: calibrated["home_win"],
        0: calibrated["draw"],
        -1: calibrated["away_win"],
    }

    def reweight(row):
        result = int(row["result"])
        raw_mass = max(raw_mass_by_result[result], 1e-9)
        return row["probability"] * calibrated_mass_by_result[result] / raw_mass

    scores["calibrated_probability"] = scores.apply(reweight, axis=1)
    scores["calibrated_probability"] = scores["calibrated_probability"] / scores["calibrated_probability"].sum()

    return scores, calibrated

## 7. Chronological Holdout Evaluation

In [ ]:
evaluation_rows = []

for _, row in test_df.iterrows():
    X_row = pd.DataFrame([row[FEATURES].to_dict()]).apply(pd.to_numeric, errors="coerce").astype(float)
    home_lambda, away_lambda = predict_lambdas_from_features(X_row)
    scores, calibrated_probs = calibrated_score_distribution(home_lambda, away_lambda)

    pred_result = max(
        {1: calibrated_probs["home_win"], 0: calibrated_probs["draw"], -1: calibrated_probs["away_win"]},
        key={1: calibrated_probs["home_win"], 0: calibrated_probs["draw"], -1: calibrated_probs["away_win"]}.get,
    )
    actual_result = result_from_scores(row["home_score"], row["away_score"])
    best_score = scores.sort_values("calibrated_probability", ascending=False).iloc[0]

    evaluation_rows.append({
        "date": row["date"],
        "home_team": row["home_team"],
        "away_team": row["away_team"],
        "actual_home_score": row["home_score"],
        "actual_away_score": row["away_score"],
        "predicted_home_score": int(best_score["home_score"]),
        "predicted_away_score": int(best_score["away_score"]),
        "actual_result": actual_result,
        "predicted_result": pred_result,
        "predicted_confidence": max(calibrated_probs.values()),
    })

evaluation_df = pd.DataFrame(evaluation_rows)

exact_score_accuracy = (
    (evaluation_df["actual_home_score"] == evaluation_df["predicted_home_score"])
    & (evaluation_df["actual_away_score"] == evaluation_df["predicted_away_score"])
).mean()

result_accuracy = (evaluation_df["actual_result"] == evaluation_df["predicted_result"]).mean()

print("Exact score accuracy:", exact_score_accuracy)
print("Result accuracy:", result_accuracy)
display(evaluation_df.head())

## 8. Calibration Diagnostics + Feature Importance

In [ ]:
calibration_check = evaluation_df.copy()
calibration_check["prediction_correct"] = (calibration_check["actual_result"] == calibration_check["predicted_result"]).astype(int)
calibration_check["confidence_bin"] = pd.cut(
    calibration_check["predicted_confidence"],
    bins=np.linspace(0, 1, 11),
    include_lowest=True,
)

calibration_summary = (
    calibration_check
    .groupby("confidence_bin", observed=False)
    .agg(
        matches=("prediction_correct", "size"),
        average_confidence=("predicted_confidence", "mean"),
        observed_accuracy=("prediction_correct", "mean"),
    )
    .reset_index()
)

print("Calibration table: predicted confidence vs observed accuracy")
display(calibration_summary)

if hasattr(home_model, "feature_importances_"):
    feature_importance = pd.DataFrame({
        "feature": FEATURES,
        "home_importance": home_model.feature_importances_,
        "away_importance": away_model.feature_importances_,
    })
    feature_importance["total_importance"] = feature_importance["home_importance"] + feature_importance["away_importance"]
    feature_importance = feature_importance.sort_values("total_importance", ascending=False).reset_index(drop=True)
    display(feature_importance)
else:
    print("Feature importance is not available for this model type.")

## 9. Future Match Prediction Functions

In [ ]:
def make_future_feature_row(home_team, away_team, neutral=1, importance=5):
    available = set(team_strength_dynamic["team"])

    if home_team not in available:
        raise ValueError(f"Missing home team in team_strength_dynamic: {home_team}")
    if away_team not in available:
        raise ValueError(f"Missing away team in team_strength_dynamic: {away_team}")

    home = team_strength_dynamic.loc[team_strength_dynamic["team"] == home_team].iloc[0]
    away = team_strength_dynamic.loc[team_strength_dynamic["team"] == away_team].iloc[0]

    row = {
        "elo_diff": home["dynamic_elo"] - away["dynamic_elo"],
        "form_diff": home["form_10"] - away["form_10"],
        "attack_diff": home["attack_20"] - away["attack_20"],
        "defense_diff": away["defense_20"] - home["defense_20"],
        "neutral": int(neutral),
        "importance": int(importance),
        "conf_strength_diff": home["conf_strength"] - away["conf_strength"],
        "host_diff": int(home["is_2026_host"]) - int(away["is_2026_host"]),
    }

    return pd.DataFrame([row])[FEATURES].astype(float)


def predict_match(home_team, away_team, neutral=1, importance=5):
    X_row = make_future_feature_row(home_team, away_team, neutral=neutral, importance=importance)
    home_lambda, away_lambda = predict_lambdas_from_features(X_row)
    scores, calibrated_probs = calibrated_score_distribution(home_lambda, away_lambda)
    best_score = scores.sort_values("calibrated_probability", ascending=False).iloc[0]

    return {
        "home_team": home_team,
        "away_team": away_team,
        "home_xg": home_lambda,
        "away_xg": away_lambda,
        "home_win": calibrated_probs["home_win"],
        "draw": calibrated_probs["draw"],
        "away_win": calibrated_probs["away_win"],
        "predicted_score": f"{int(best_score['home_score'])}-{int(best_score['away_score'])}",
        "score_distribution": scores,
    }

prediction = predict_match("Spain", "France")
print(prediction["home_team"], "vs", prediction["away_team"])
print("xG:", round(prediction["home_xg"], 2), "-", round(prediction["away_xg"], 2))
print("Predicted score:", prediction["predicted_score"])
print("W/D/L:", round(prediction["home_win"], 3), round(prediction["draw"], 3), round(prediction["away_win"], 3))

## 10. 2026 Groups

In [ ]:
# Replace these with the official draw when you have it.
groups = {
    "A": ["Mexico", "South Africa", "South Korea", "Czechoslovakia"],
    "B": ["Canada", "Qatar", "Switzerland", "Bosnia and Herzegovina"],
    "C": ["Brazil", "Morocco", "Scotland", "Haiti"],
    "D": ["USA", "Paraguay", "Australia", "Turkey"],
    "E": ["Germany", "Ivory Coast", "Ecuador", "Curacao"],
    "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "H": ["Spain", "Cape Verde", "Saudi Arabia", "Uruguay"],
    "I": ["France", "Senegal", "Norway", "Iraq"],
    "J": ["Argentina", "Algeria", "Austria", "Jordan"],
    "K": ["Portugal", "DR Congo", "Uzbekistan", "Colombia"],
    "L": ["England", "Croatia", "Ghana", "Panama"],
}

missing = sorted({team for teams in groups.values() for team in teams} - set(team_strength_dynamic["team"]))
if missing:
    raise ValueError(f"Missing group teams in team_strength_dynamic: {missing}")

print("Teams:", sum(len(v) for v in groups.values()))

## 11. Live Tournament Results Layer

Use this section once the World Cup starts. Add completed match results here, and the simulator will keep those scores fixed while still simulating all unplayed matches.

The results are saved to `data/actual_2026_results.csv`, so you can rerun the notebook without losing entered scores.

In [ ]:
ACTUAL_RESULTS_PATH = DATA_DIR / "actual_2026_results.csv"

ACTUAL_RESULTS_COLUMNS = [
    "stage",          # "group" or "knockout"
    "group",          # Group letter for group-stage matches, otherwise blank
    "round",          # Knockout round name, otherwise blank
    "match_id",       # Official knockout match id when known, otherwise blank
    "home_team",
    "away_team",
    "home_goals",
    "away_goals",
    "winner",         # Required only if a knockout match is tied on goals after the recorded score
]


def load_actual_results(path=ACTUAL_RESULTS_PATH):
    if path.exists():
        df = pd.read_csv(path)
    else:
        df = pd.DataFrame(columns=ACTUAL_RESULTS_COLUMNS)

    for col in ACTUAL_RESULTS_COLUMNS:
        if col not in df.columns:
            df[col] = np.nan

    df = df[ACTUAL_RESULTS_COLUMNS].copy()

    for team_col in ["home_team", "away_team", "winner"]:
        df[team_col] = df[team_col].apply(clean_team)

    for col in ["home_goals", "away_goals"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["match_id"] = pd.to_numeric(df["match_id"], errors="coerce")

    return df


actual_results = load_actual_results()


def save_actual_results(path=ACTUAL_RESULTS_PATH):
    actual_results.to_csv(path, index=False)
    print(f"Saved {len(actual_results)} actual results to {path}")


def add_match_result(
    home_team,
    away_team,
    home_goals,
    away_goals,
    stage="group",
    group=None,
    round_name=None,
    match_id=None,
    winner=None,
    save=True,
):
    """
    Add or replace a completed World Cup match result.

    Examples:
    add_match_result("Mexico", "South Africa", 2, 1, stage="group", group="A")
    add_match_result("Spain", "France", 1, 1, stage="knockout", round_name="Final", match_id=104, winner="Spain")
    """
    global actual_results

    home_team = clean_team(home_team)
    away_team = clean_team(away_team)
    winner = clean_team(winner) if winner is not None else winner

    if stage not in {"group", "knockout"}:
        raise ValueError("stage must be either 'group' or 'knockout'")

    if stage == "group" and group is None:
        raise ValueError("Group-stage results need a group, e.g. group='A'")

    if stage == "knockout" and home_goals == away_goals and winner is None:
        raise ValueError("Knockout matches tied on goals need winner='Team Name'")

    new_row = {
        "stage": stage,
        "group": group,
        "round": round_name,
        "match_id": match_id,
        "home_team": home_team,
        "away_team": away_team,
        "home_goals": int(home_goals),
        "away_goals": int(away_goals),
        "winner": winner,
    }

    # Replace existing entry for the same match.
    if match_id is not None and not pd.isna(match_id):
        mask = pd.to_numeric(actual_results["match_id"], errors="coerce") == int(match_id)
    else:
        same_pair = (
            ((actual_results["home_team"] == home_team) & (actual_results["away_team"] == away_team)) |
            ((actual_results["home_team"] == away_team) & (actual_results["away_team"] == home_team))
        )
        mask = (actual_results["stage"] == stage) & same_pair
        if group is not None:
            mask = mask & (actual_results["group"] == group)
        if round_name is not None:
            mask = mask & (actual_results["round"] == round_name)

    actual_results = actual_results.loc[~mask].copy()
    actual_results = pd.concat([actual_results, pd.DataFrame([new_row])], ignore_index=True)
    actual_results = actual_results[ACTUAL_RESULTS_COLUMNS]

    if save:
        save_actual_results()

    return actual_results.tail(1)


def clear_actual_results(save=True):
    """Clear all entered live results. Useful before testing a new scenario."""
    global actual_results
    actual_results = pd.DataFrame(columns=ACTUAL_RESULTS_COLUMNS)
    if save:
        save_actual_results()
    return actual_results


def find_actual_result(home_team, away_team, stage=None, group=None, round_name=None, match_id=None):
    """Find a real result and orient the score to the requested home/away order."""
    if actual_results.empty:
        return None

    home_team = clean_team(home_team)
    away_team = clean_team(away_team)
    df = actual_results.copy()

    if match_id is not None:
        df = df[pd.to_numeric(df["match_id"], errors="coerce") == int(match_id)]
    else:
        df = df[
            ((df["home_team"] == home_team) & (df["away_team"] == away_team)) |
            ((df["home_team"] == away_team) & (df["away_team"] == home_team))
        ]

    if stage is not None:
        df = df[df["stage"] == stage]
    if group is not None:
        df = df[df["group"] == group]
    if round_name is not None:
        df = df[df["round"] == round_name]

    if df.empty:
        return None

    row = df.iloc[-1]
    stored_home = row["home_team"]
    stored_away = row["away_team"]
    stored_home_goals = int(row["home_goals"])
    stored_away_goals = int(row["away_goals"])

    if stored_home == home_team and stored_away == away_team:
        oriented_home_goals = stored_home_goals
        oriented_away_goals = stored_away_goals
    elif stored_home == away_team and stored_away == home_team:
        oriented_home_goals = stored_away_goals
        oriented_away_goals = stored_home_goals
    else:
        return None

    winner = row["winner"] if pd.notna(row["winner"]) and str(row["winner"]).strip() else None
    if winner is None:
        if oriented_home_goals > oriented_away_goals:
            winner = home_team
        elif oriented_home_goals < oriented_away_goals:
            winner = away_team
        else:
            winner = None

    return {
        "home_team": home_team,
        "away_team": away_team,
        "home_goals": oriented_home_goals,
        "away_goals": oriented_away_goals,
        "winner": winner,
        "source": "actual",
        "match_id": int(row["match_id"]) if pd.notna(row["match_id"]) else match_id,
        "round": row["round"] if pd.notna(row["round"]) else round_name,
        "group": row["group"] if pd.notna(row["group"]) else group,
    }


# Example usage once the tournament starts:
# add_match_result("Mexico", "South Africa", 2, 1, stage="group", group="A")
# add_match_result("Spain", "France", 1, 1, stage="knockout", round_name="Final", match_id=104, winner="Spain")

print(f"Loaded {len(actual_results)} actual results")
display(actual_results)


In [ ]:
#////
# Live Tournament update


#////

## 12. Calibrated Score-Based Group Simulation

In [ ]:
def sample_score(home_team, away_team, neutral=1, importance=5, max_goals=10):
    X_row = make_future_feature_row(home_team, away_team, neutral=neutral, importance=importance)
    home_lambda, away_lambda = predict_lambdas_from_features(X_row)
    scores, calibrated_probs = calibrated_score_distribution(home_lambda, away_lambda, max_goals=max_goals)

    idx = rng.choice(scores.index.to_numpy(), p=scores["calibrated_probability"].to_numpy())
    selected = scores.loc[idx]

    return {
        "home_team": home_team,
        "away_team": away_team,
        "home_goals": int(selected["home_score"]),
        "away_goals": int(selected["away_score"]),
        "home_xg": home_lambda,
        "away_xg": away_lambda,
        "home_win_probability": calibrated_probs["home_win"],
        "draw_probability": calibrated_probs["draw"],
        "away_win_probability": calibrated_probs["away_win"],
        "source": "simulated",
    }


def simulate_match(home_team, away_team, neutral=1, knockout=False, importance=5, group=None, round_name=None, match_id=None):
    actual = find_actual_result(
        home_team,
        away_team,
        stage="knockout" if knockout else "group",
        group=group,
        round_name=round_name,
        match_id=match_id,
    )

    if actual is not None:
        # Keep model probabilities attached for reference, but lock the actual score.
        X_row = make_future_feature_row(home_team, away_team, neutral=neutral, importance=importance)
        home_lambda, away_lambda = predict_lambdas_from_features(X_row)
        _, calibrated_probs = calibrated_score_distribution(home_lambda, away_lambda)
        actual.update({
            "home_xg": home_lambda,
            "away_xg": away_lambda,
            "home_win_probability": calibrated_probs["home_win"],
            "draw_probability": calibrated_probs["draw"],
            "away_win_probability": calibrated_probs["away_win"],
        })
        return actual

    match = sample_score(home_team, away_team, neutral=neutral, importance=importance)
    match["match_id"] = match_id
    match["round"] = round_name
    match["group"] = group

    if knockout and match["home_goals"] == match["away_goals"]:
        # Penalties/extra time fallback. Future improvement: dedicated shootout model.
        probs = np.array([match["home_win_probability"], match["away_win_probability"]], dtype=float)
        probs = probs / probs.sum() if probs.sum() > 0 else np.array([0.5, 0.5])
        match["winner"] = rng.choice([home_team, away_team], p=probs)
    else:
        match["winner"] = None if match["home_goals"] == match["away_goals"] else (home_team if match["home_goals"] > match["away_goals"] else away_team)

    return match


def init_group_table(teams):
    return {
        team: {"team": team, "played": 0, "wins": 0, "draws": 0, "losses": 0, "gf": 0, "ga": 0, "gd": 0, "points": 0}
        for team in teams
    }


def update_table(table, match):
    home = match["home_team"]
    away = match["away_team"]
    home_goals = match["home_goals"]
    away_goals = match["away_goals"]

    table[home]["played"] += 1
    table[away]["played"] += 1

    table[home]["gf"] += home_goals
    table[home]["ga"] += away_goals
    table[away]["gf"] += away_goals
    table[away]["ga"] += home_goals

    table[home]["gd"] = table[home]["gf"] - table[home]["ga"]
    table[away]["gd"] = table[away]["gf"] - table[away]["ga"]

    if home_goals > away_goals:
        table[home]["wins"] += 1
        table[away]["losses"] += 1
        table[home]["points"] += 3
    elif home_goals < away_goals:
        table[away]["wins"] += 1
        table[home]["losses"] += 1
        table[away]["points"] += 3
    else:
        table[home]["draws"] += 1
        table[away]["draws"] += 1
        table[home]["points"] += 1
        table[away]["points"] += 1


def simulate_group(teams, group_name=None):
    table = init_group_table(teams)
    matches_out = []

    for home, away in combinations(teams, 2):
        match = simulate_match(home, away, neutral=1, knockout=False, importance=5, group=group_name)
        matches_out.append(match)
        update_table(table, match)

    standings = pd.DataFrame(table.values()).sort_values(["points", "gd", "gf"], ascending=False).reset_index(drop=True)
    return standings, pd.DataFrame(matches_out)


def simulate_group_stage(groups):
    all_standings = []
    all_matches = []
    qualified = []
    third_place_rows = []

    for group_name, teams in groups.items():
        standings, matches_df = simulate_group(teams, group_name=group_name)
        standings["group"] = group_name
        matches_df["group"] = group_name

        all_standings.append(standings)
        all_matches.append(matches_df)

        qualified.extend(standings.iloc[0:2]["team"].tolist())
        third_place_rows.append(standings.iloc[2])

    third_place_df = pd.DataFrame(third_place_rows)
    best_thirds = third_place_df.sort_values(["points", "gd", "gf"], ascending=False).head(8)
    qualified.extend(best_thirds["team"].tolist())

    return {
        "standings": pd.concat(all_standings).reset_index(drop=True),
        "matches": pd.concat(all_matches).reset_index(drop=True),
        "qualified": qualified,
        "best_thirds": best_thirds.reset_index(drop=True),
    }


## 13. Official-Style 2026 Knockout Bracket

In [ ]:
ROUND_OF_32_TEMPLATE_2026 = [
    (73, ("runner_up", "A"), ("runner_up", "B")),
    (74, ("winner", "C"), ("runner_up", "F")),
    (75, ("winner", "E"), ("third", ("A", "B", "C", "D", "F"))),
    (76, ("winner", "F"), ("runner_up", "C")),
    (77, ("runner_up", "E"), ("runner_up", "I")),
    (78, ("winner", "I"), ("third", ("C", "D", "F", "G", "H"))),
    (79, ("winner", "A"), ("third", ("C", "E", "F", "H", "I"))),
    (80, ("winner", "L"), ("third", ("E", "H", "I", "J", "K"))),
    (81, ("winner", "G"), ("third", ("A", "E", "H", "I", "J"))),
    (82, ("winner", "D"), ("third", ("B", "E", "F", "I", "J"))),
    (83, ("winner", "H"), ("runner_up", "J")),
    (84, ("runner_up", "K"), ("runner_up", "L")),
    (85, ("winner", "B"), ("third", ("E", "F", "G", "I", "J"))),
    (86, ("runner_up", "D"), ("runner_up", "G")),
    (87, ("winner", "J"), ("runner_up", "H")),
    (88, ("winner", "K"), ("third", ("D", "E", "I", "J", "L"))),
]

ROUND_OF_16_TEMPLATE_2026 = [(89, 73, 75), (90, 74, 77), (91, 76, 78), (92, 79, 80), (93, 83, 84), (94, 81, 82), (95, 86, 88), (96, 85, 87)]
QUARTER_FINAL_TEMPLATE_2026 = [(97, 89, 90), (98, 93, 94), (99, 91, 92), (100, 95, 96)]
SEMI_FINAL_TEMPLATE_2026 = [(101, 97, 98), (102, 99, 100)]
FINAL_TEMPLATE_2026 = [(104, 101, 102)]


def standings_positions_by_group(stage):
    positions = {}
    third_place_by_group = {}

    for group_name in sorted(groups.keys()):
        group_table = (
            stage["standings"]
            .loc[stage["standings"]["group"] == group_name]
            .sort_values(["points", "gd", "gf"], ascending=False)
            .reset_index(drop=True)
        )
        positions[("winner", group_name)] = group_table.iloc[0]["team"]
        positions[("runner_up", group_name)] = group_table.iloc[1]["team"]
        third_place_by_group[group_name] = group_table.iloc[2]["team"]

    qualified_third_groups = set(stage["best_thirds"]["group"].tolist())
    qualified_thirds = {group: third_place_by_group[group] for group in qualified_third_groups}
    return positions, qualified_thirds


def assign_third_place_slots_2026(qualified_thirds):
    third_slots = []

    for match_id, team_1_slot, team_2_slot in ROUND_OF_32_TEMPLATE_2026:
        for side, slot in [("team_1", team_1_slot), ("team_2", team_2_slot)]:
            if slot[0] == "third":
                third_slots.append({"match_id": match_id, "side": side, "eligible_groups": tuple(slot[1])})

    qualified_groups = set(qualified_thirds.keys())
    third_slots = sorted(third_slots, key=lambda s: len(set(s["eligible_groups"]) & qualified_groups))

    assignment = {}
    used_groups = set()

    def backtrack(index):
        if index == len(third_slots):
            return True

        slot = third_slots[index]
        candidates = [g for g in slot["eligible_groups"] if g in qualified_groups and g not in used_groups]

        for group in sorted(candidates):
            used_groups.add(group)
            assignment[(slot["match_id"], slot["side"])] = qualified_thirds[group]
            if backtrack(index + 1):
                return True
            used_groups.remove(group)
            assignment.pop((slot["match_id"], slot["side"]), None)

        return False

    if not backtrack(0):
        raise ValueError(f"Could not assign third-place teams. Qualified groups: {sorted(qualified_groups)}")

    return assignment


def build_round_of_32_official_2026(stage):
    positions, qualified_thirds = standings_positions_by_group(stage)
    third_assignment = assign_third_place_slots_2026(qualified_thirds)
    fixtures = []

    for match_id, team_1_slot, team_2_slot in ROUND_OF_32_TEMPLATE_2026:
        team_1 = third_assignment[(match_id, "team_1")] if team_1_slot[0] == "third" else positions[team_1_slot]
        team_2 = third_assignment[(match_id, "team_2")] if team_2_slot[0] == "third" else positions[team_2_slot]
        fixtures.append({"match_id": match_id, "home_team": team_1, "away_team": team_2})

    return fixtures


def simulate_knockout_match(match_id, home_team, away_team, round_name):
    match = simulate_match(
        home_team,
        away_team,
        neutral=1,
        knockout=True,
        importance=5,
        round_name=round_name,
        match_id=match_id,
    )
    match["match_id"] = match_id
    match["round"] = round_name
    return match


def simulate_knockout_from_template(previous_winners, template, round_name):
    results = []
    winners = {}

    for match_id, prev_a, prev_b in template:
        home_team = previous_winners[prev_a]
        away_team = previous_winners[prev_b]
        match = simulate_knockout_match(match_id, home_team, away_team, round_name)
        results.append(match)
        winners[match_id] = match["winner"]

    return winners, pd.DataFrame(results)


def simulate_official_2026_knockout(stage):
    all_results = []
    winners_by_match = {}

    r32_results = []
    for fixture in build_round_of_32_official_2026(stage):
        match = simulate_knockout_match(fixture["match_id"], fixture["home_team"], fixture["away_team"], "Round of 32")
        r32_results.append(match)
        winners_by_match[fixture["match_id"]] = match["winner"]

    all_results.append(pd.DataFrame(r32_results))

    for template, round_name in [
        (ROUND_OF_16_TEMPLATE_2026, "Round of 16"),
        (QUARTER_FINAL_TEMPLATE_2026, "Quarter-final"),
        (SEMI_FINAL_TEMPLATE_2026, "Semi-final"),
        (FINAL_TEMPLATE_2026, "Final"),
    ]:
        winners, round_results = simulate_knockout_from_template(winners_by_match, template, round_name)
        winners_by_match.update(winners)
        all_results.append(round_results)

    champion = winners_by_match[104]
    knockout_results = pd.concat(all_results).reset_index(drop=True)
    return champion, knockout_results

## 14. Final Monte Carlo Simulation

In [ ]:
def simulate_tournament(groups):
    stage = simulate_group_stage(groups)
    champion, knockout_results = simulate_official_2026_knockout(stage)

    reached = defaultdict(set)

    for team in stage["qualified"]:
        reached["round_of_32"].add(team)

    next_round_by_round_name = {
        "Round of 32": "round_of_16",
        "Round of 16": "quarter_final",
        "Quarter-final": "semi_final",
        "Semi-final": "final",
        "Final": "champion",
    }

    for round_name, next_round_name in next_round_by_round_name.items():
        round_matches = knockout_results[knockout_results["round"] == round_name]
        for winner in round_matches["winner"]:
            reached[next_round_name].add(winner)

    return champion, reached, {
        "champion": champion,
        "group_standings": stage["standings"],
        "group_matches": stage["matches"],
        "best_thirds": stage["best_thirds"],
        "knockout_results": knockout_results,
    }

round_counts = defaultdict(Counter)
champions = []
example_tournament = None

for i in tqdm(range(N_SIMULATIONS)):
    champion, reached, tournament = simulate_tournament(groups)
    champions.append(champion)

    if i == 0:
        example_tournament = tournament

    for round_name, teams in reached.items():
        for team in teams:
            round_counts[round_name][team] += 1

all_teams = sorted({team for teams in groups.values() for team in teams})
round_probability_rows = []

for team in all_teams:
    row = {"team": team}
    for round_name in ["round_of_32", "round_of_16", "quarter_final", "semi_final", "final", "champion"]:
        row[round_name] = round_counts[round_name][team] / N_SIMULATIONS * 100
    round_probability_rows.append(row)

round_probs = (
    pd.DataFrame(round_probability_rows)
    .sort_values("champion", ascending=False)
    .reset_index(drop=True)
)

round_probs.to_csv(DATA_DIR / "world_cup_round_probabilities_final.csv", index=False)

print("Monte Carlo simulations:", N_SIMULATIONS)
display(round_probs.head(20))
print("Example champion:", example_tournament["champion"])
display(example_tournament["best_thirds"])
display(example_tournament["knockout_results"])

## 15. Predicted Group Match Scores

These are deterministic most-likely scores from the calibrated score distribution, not simulated results.

In [ ]:
all_match_predictions = []

for group_name, teams in groups.items():
    for home, away in combinations(teams, 2):
        pred = predict_match(home, away, neutral=1, importance=5)
        all_match_predictions.append({
            "group": group_name,
            "home_team": home,
            "away_team": away,
            "home_xg": pred["home_xg"],
            "away_xg": pred["away_xg"],
            "predicted_score": pred["predicted_score"],
            "home_win_probability": pred["home_win"],
            "draw_probability": pred["draw"],
            "away_win_probability": pred["away_win"],
        })

predicted_group_scores = pd.DataFrame(all_match_predictions).sort_values(["group", "home_team", "away_team"])
predicted_group_scores.to_csv(DATA_DIR / "predicted_group_match_scores_final.csv", index=False)
display(predicted_group_scores)